# SmolVLA LoRA Training

Fine-tune SmolVLA (450M) with LoRA on LIBERO-Object. Optimized for Kaggle T4 16GB.

In [ ]:
# Cell 1: Install and setup
!pip install -q lerobot[smolvla,peft,libero]
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 2: Train SmolVLA with LoRA
import subprocess
import time

start = time.time()
result = subprocess.run([
    "lerobot-train",
    "--policy.path=lerobot/smolvla_base",
    "--dataset.repo_id=HuggingFaceVLA/libero",
    "--batch_size=8",
    "--steps=20000",
    "--save_checkpoint=true",
    "--log_interval=100",
    "--peft.method_type=LORA",
    "--peft.r=32",
    "--policy.optimizer_lr=1e-3",
    "--policy.scheduler_decay_lr=1e-4",
    "--env.type=libero",
    "--env.task=libero_object",
    "--wandb.enable=false",
    "--policy.output_features=null",
    "--policy.input_features=null",
], capture_output=False, text=True, timeout=36000)  # 10h timeout

elapsed = time.time() - start
print(f"\nTraining completed in {elapsed/3600:.1f} hours")

In [ ]:
# Cell 3: Save and push checkpoint
import glob
from huggingface_hub import HfApi

# Find latest checkpoint
checkpoints = sorted(glob.glob("outputs/*/checkpoints/*"))
print(f"Checkpoints found: {checkpoints}")

if checkpoints:
    latest = checkpoints[-1]
    print(f"Latest checkpoint: {latest}")

    # Optional: push to Hub (set your HF token first)
    # api = HfApi()
    # api.upload_folder(
    #     folder_path=latest,
    #     repo_id="YOUR_USERNAME/smolvla-libero-object-lora",
    #     repo_type="model"
    # )
    # print("Pushed to Hub!")

In [ ]:
# Cell 4: Plot training loss from logs
import json
import matplotlib.pyplot as plt

# LeRobot logs training metrics to stdout/files — parse them
# Location depends on LeRobot version; adapt path as needed
log_files = glob.glob("outputs/*/logs/*.json") + glob.glob("outputs/*/train_log.jsonl")
print(f"Log files: {log_files}")

if log_files:
    losses = []
    steps = []
    with open(log_files[0]) as f:
        for line in f:
            try:
                entry = json.loads(line)
                if "loss" in entry:
                    losses.append(entry["loss"])
                    steps.append(entry.get("step", len(steps)))
            except json.JSONDecodeError:
                continue

    if losses:
        plt.figure(figsize=(10, 4))
        plt.plot(steps, losses)
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("SmolVLA LoRA Training Loss")
        plt.grid(True, alpha=0.3)
        plt.savefig("training_loss.png", dpi=100)
        plt.show()
        print(f"Final loss: {losses[-1]:.4f}")